# Qwen3-Embedding-0.6B + Binary Classifier
## Train · Register · Deploy

**Workflow**
1. Generate / load a small labelled dataset
2. Train (or load pre-trained) classification head weights
3. Log the custom MLflow model (`Qwen3ClassifierModel`) to the MLflow registry
4. Deploy the registered model to a Databricks Model Serving endpoint (GPU)
   using the Databricks Python SDK

### 0  Install / verify dependencies

In [0]:
# %pip install vllm==0.9.1 sentence-transformers==4.1.0 mlflow[databricks]>=3.12.0 databricks-sdk>=0.28 torch==2.6.0 torchvision==0.21.0  --quiet
%pip install --upgrade -r requirements.txt

In [0]:
dbutils.library.restartPython()

### 1  Configuration

In [0]:
import os

# ------------------------------------------------------------------
# Databricks / Unity Catalog
# ------------------------------------------------------------------
CATALOG       = "uc_sriharsha_jana"              # UC catalog
DB_SCHEMA     = "default"           # UC schema
MODEL_NAME    = "qwen3_binary_classifier"
FULL_MODEL_NAME = f"{CATALOG}.{DB_SCHEMA}.{MODEL_NAME}"

# Serving endpoint
ENDPOINT_NAME = "qwen3-binary-classifier-endpoint"

# Local paths (relative to repo root when running in a DAB job)
REPO_ROOT         = os.environ.get("BUNDLE_ROOT", "/Workspace/Users/sriharsha.jana@databricks.com/customers/angelone/custom-embedding")
MODEL_SRC_PATH    = f"{REPO_ROOT}/qwen3_classifier_model.py"
WEIGHTS_LOCAL_DIR = "/tmp/qwen3_classifier"

# MLflow experiment
EXPERIMENT_NAME = f"/Users/{dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()}/qwen3_binary_classifier"

print(f"Full model name : {FULL_MODEL_NAME}")
print(f"Endpoint        : {ENDPOINT_NAME}")
print(f"Experiment      : {EXPERIMENT_NAME}")

# This is to disable the FIPS mode from openssl as it was resulting in python kernel crashes
os.environ.pop("OPENSSL_FORCE_FIPS_MODE", None)

#setup the logging level for vllm
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"

### 2  Create / load classification head weights

In [0]:
import os, torch, torch.nn as nn
import numpy as np
from pathlib import Path

os.makedirs(WEIGHTS_LOCAL_DIR, exist_ok=True)
WEIGHTS_PATH = os.path.join(WEIGHTS_LOCAL_DIR, "classification_weights.pt")

# -----------------------------------------------------------------
# If you have pre-trained weights, copy them to WEIGHTS_PATH and
# skip to the next cell.
# -----------------------------------------------------------------
# For demonstration we train a tiny head on random synthetic data.
# Replace this block with your real training loop.
# -----------------------------------------------------------------

EMBEDDING_DIM = 1024   # Qwen3-Embedding-0.6B output dim
N_TRAIN       = 200

torch.manual_seed(42)

# Synthetic embeddings + binary labels
X = torch.randn(N_TRAIN, EMBEDDING_DIM)
y = torch.randint(0, 2, (N_TRAIN,))

head = nn.Linear(EMBEDDING_DIM, 2)
optimizer = torch.optim.Adam(head.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

head.train()
for epoch in range(20):
    optimizer.zero_grad()
    loss = criterion(head(X), y)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"  epoch {epoch+1:2d}  loss={loss.item():.4f}")

torch.save(head.state_dict(), WEIGHTS_PATH)
print(f"\nWeights saved to {WEIGHTS_PATH}")

### 3  Log model to MLflow

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

In [0]:
# Pip environment captured inside the logged model artifact
with open("requirements.txt") as f:
    logged_model_pip = [line.strip() for line in f if line.strip() and not line.startswith("#")]
logged_model_pip

In [0]:
import mlflow
import pandas as pd
from mlflow.models import ModelSignature
from mlflow.types.schema import Schema, ColSpec

MODEL_PATH = os.path.join(os.getcwd(), "qwen3_classifier_model.py")
print(MODEL_PATH)

input_schema = Schema([ColSpec("string", "text")])
output_schema = Schema([ColSpec("integer")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

with mlflow.start_run(run_name="qwen3_binary_classifier") as run:
    mlflow.log_params({
        "base_model": "Qwen/Qwen3-Embedding-0.6B",
        "embedding_dim": 1024,
        "head_architecture": "Linear(1024, 2)",
        "training_samples": N_TRAIN,
    })

    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=MODEL_PATH,
        artifacts={
            "model_dir": WEIGHTS_LOCAL_DIR,   # directory containing classification_weights.pt
        },
        pip_requirements=logged_model_pip,
        signature=signature,
        input_example=pd.DataFrame({"text": ["example input text"]}),
        registered_model_name=FULL_MODEL_NAME,
    )

    run_id = run.info.run_id
    print(f"Run ID : {run_id}")
    print(f"Model  : {FULL_MODEL_NAME}")

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_version = model_info.registered_model_version # Adjust if you want a specific stage/version

client.set_registered_model_alias(name=FULL_MODEL_NAME, version=model_version, alias="prod")

In [0]:
import subprocess
import os

# List processes using GPU
gpu_processes = subprocess.check_output(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"]).decode().strip().split('\n')
gpu_pids = [pid for pid in gpu_processes if pid.isdigit()]

print("GPU PIDs:", gpu_pids)

# Kill each PID
for pid in gpu_pids:
    try:
        os.kill(int(pid), 9)
        print(f"Killed PID {pid}")
    except Exception as e:
        print(f"Failed to kill PID {pid}: {e}")

### 4  Quick smoke-test (load & predict locally)

In [0]:
dbutils.library.restartPython()

In [0]:
import os

#setup the logging level for vllm
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"

# This is to disable the FIPS mode from openssl as it was resulting in python kernel crashes
os.environ.pop("OPENSSL_FORCE_FIPS_MODE", None)

In [0]:
import mlflow
import pandas as pd

CATALOG       = "uc_sriharsha_jana"              # UC catalog
DB_SCHEMA     = "default"           # UC schema
MODEL_NAME    = "qwen3_binary_classifier"
FULL_MODEL_NAME = f"{CATALOG}.{DB_SCHEMA}.{MODEL_NAME}"

model_uri = f"models:/{FULL_MODEL_NAME}@prod"
loaded_model = mlflow.pyfunc.load_model(model_uri)

In [0]:
test_inputs = pd.DataFrame({
    "text": [
        "The quarterly earnings exceeded analyst expectations.",
        "System failure detected in production cluster.",
        "New regulatory guidelines for financial institutions.",
        "Critical security vulnerability found in open-source library.",
    ]
})

predictions = loaded_model.predict(test_inputs)
print("Predictions:", predictions)

for text, pred in zip(test_inputs["text"], predictions):
    label = "POSITIVE (1)" if pred == 1 else "NEGATIVE (0)"
    print(f"  [{label}] {text[:70]}")

### 5  Deploy to Databricks Model Serving (GPU)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import ModelVersionInfoStatus

wc = WorkspaceClient()

versions = wc.model_versions.list(full_name=FULL_MODEL_NAME)
version_list = list(versions)
latest_version = max(version_list, key=lambda v: int(v.version))
model_version  = latest_version.version

print(f"Deploying version {model_version} of {FULL_MODEL_NAME}")

In [0]:
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedModelInput,
    ServedModelInputWorkloadType,
)
from databricks.sdk.errors import ResourceDoesNotExist

ENDPOINT_NAME = "qwen3-binary-classifier-endpoint"

endpoint_config = EndpointCoreConfigInput(
    served_models=[
        ServedModelInput(
            name=f"{MODEL_NAME}",
            model_name=FULL_MODEL_NAME,
            model_version=str(model_version),
            workload_size="Small",
            workload_type=ServedModelInputWorkloadType.GPU_MEDIUM,  # A10G
            scale_to_zero_enabled=True,
        )
    ]
)

# Check if endpoint already exists
try:
    existing_endpoint = wc.serving_endpoints.get(name=ENDPOINT_NAME)
    print(f"Updating existing endpoint: {ENDPOINT_NAME}")
    wc.serving_endpoints.update_config(
        name=ENDPOINT_NAME,
        served_models=endpoint_config.served_models,
    )
except ResourceDoesNotExist:
    print(f"Creating new endpoint: {ENDPOINT_NAME}")
    wc.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=endpoint_config,
    )

print(f"Endpoint '{ENDPOINT_NAME}' update submitted. It will be ready in a few minutes.")

### 6  Wait for endpoint to become ready

In [0]:
import time
from databricks.sdk.service.serving import EndpointStateReady

MAX_WAIT_SECONDS = 600
poll_interval    = 20
elapsed          = 0

while elapsed < MAX_WAIT_SECONDS:
    endpoint = wc.serving_endpoints.get(name=ENDPOINT_NAME)
    state = endpoint.state.ready if endpoint.state else None
    print(f"  [{elapsed:>4}s] state={state}")

    if state == EndpointStateReady.READY:
        print(f"\nEndpoint '{ENDPOINT_NAME}' is READY.")
        break

    time.sleep(poll_interval)
    elapsed += poll_interval
else:
    print(f"WARNING: endpoint not ready after {MAX_WAIT_SECONDS}s – check the UI.")

### 7  Score the deployed endpoint

In [0]:
import json

test_payload = {
    "dataframe_records": [
        {"text": "Quarterly revenue grew 15% year-over-year."},
        {"text": "Outage reported in the primary data centre."},
    ]
}

response = wc.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=test_payload["dataframe_records"],
)

print("Endpoint response:")
print(json.dumps(response.as_dict(), indent=2))